# 01 · Data audit

**Purpose.** Get M4Raw onto this machine, check what is actually in it, and record the facts every later notebook relies on.

**Runtime.** CPU is enough. Full run: about 21 GB of downloads the first time.

**Reads.** Zenodo (first time) or the zip cache on Drive (later sessions).

**Writes to Drive** (`meta/`):

| File | Content |
| --- | --- |
| `manifest.csv` | one row per h5 file: shape, acquired phase-encode lines, k-space centre line |
| `splits.csv` | one row per subject: split and whether it is usable |
| `alignment.csv` | measured T1→T2 shift per subject |
| `audit_summary.json` | the headline numbers, including the averaging-domain recommendation |

**Week 1 checkpoint covered here:** loader returns correct k-space; alignment numbers are recorded.

## 0. Setup
Mounts Drive, gets the latest project code from GitHub and makes `t1t2` importable. The same cell opens every notebook.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/leo-oxu/eval-t1-guided-t2-rec.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/eval-t1-guided-t2-rec"
    if os.path.isdir(REPO):
        if subprocess.run(["git", "-C", REPO, "pull", "-q"]).returncode:
            print("WARNING: git pull failed (local edits?). Using the code already on this VM.")
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO}/requirements.txt"], check=True)
else:
    REPO = os.path.abspath("..")
if REPO not in sys.path:
    sys.path.insert(0, REPO)

try:   # picks up edits to t1t2/ without a restart; broken in some Colab images (Python 3.13)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as e:
    print(f"autoreload unavailable ({type(e).__name__}); restart the runtime after editing t1t2/")
import t1t2
commit = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"t1t2 {t1t2.__version__} @ {commit}")

## 1. Settings
`SMOKE = True` is a quick trial: validation split only, first 3 subjects, outputs under `smoke/` so nothing real is overwritten. Set it to `False` for the real run.

In [ ]:
SMOKE = True
DRIVE_ROOT = "/content/drive/MyDrive/t1t2"   # shared project folder on Drive
KEEP_ZIPS_ON_DRIVE = True                    # ~21 GB on Drive; saves re-downloading from Zenodo
N_PHASE_CHECK = 3                            # subjects used for the repetition phase-stability check

SPLITS = ["val"] if SMOKE else ["train", "val", "test"]
MAX_SUBJECTS = 3 if SMOKE else None

In [ ]:
import json, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from t1t2 import m4raw
from t1t2.paths import get_paths
from t1t2.fft import ifft2c, kspace_to_rss
from t1t2.align import volume_shift, estimate_shift, apply_shift, edge_map
from t1t2.preprocess import head_mask

paths = get_paths(DRIVE_ROOT if IN_COLAB else None, smoke=SMOKE)
free_gb = shutil.disk_usage(paths.drive).free / 1e9
print(f"outputs -> {paths.meta}\nDrive free space: {free_gb:.1f} GB")

## 2. Get the data
Only the T1 and T2 files are unzipped (FLAIR is skipped). The motion-corrupted subjects are in a separate zip that is never downloaded, which is how they are excluded.

In [ ]:
for split in SPLITS:
    m4raw.fetch_split(split, paths, keep_zip_on_drive=KEEP_ZIPS_ON_DRIVE)

index = m4raw.index_files(paths.raw)
index = index[index.split.isin(SPLITS)]
if MAX_SUBJECTS:
    keep = index.groupby("split").scan_id.unique().apply(lambda ids: list(ids[:MAX_SUBJECTS]))
    index = index[index.apply(lambda r: r.scan_id in keep[r.split], axis=1)]
index = index.reset_index(drop=True)
print(f"{len(index)} files, {index.scan_id.nunique()} subjects")
index.groupby(["split", "contrast"]).agg(files=("path", "size"), subjects=("scan_id", "nunique"))

## 3. Manifest: shapes and zero-padded phase-encode lines
M4Raw stores 256 phase-encode (PE) lines but only about 195 are acquired; the rest are zeros. For every file we record which lines hold data and where the k-space centre is. Undersampling masks must be built on the acquired lines only.

In [ ]:
rows = []
for r in tqdm(index.itertuples(), total=len(index)):
    a = m4raw.load_attrs(r.path)
    n_slices, n_coils, n_fe, n_pe = a["kspace_shape"]
    k = m4raw.load_kspace(r.path, slices=n_slices // 2)          # middle slice is enough
    acq = m4raw.acquired_pe(k)
    lines = np.flatnonzero(acq)
    rows.append(dict(split=r.split, scan_id=r.scan_id, contrast=r.contrast, rep=r.rep,
                     file=os.path.basename(r.path), n_slices=n_slices, n_coils=n_coils,
                     n_fe=n_fe, n_pe=n_pe, n_pe_acquired=int(acq.sum()),
                     pe_first=int(lines[0]), pe_last=int(lines[-1]),
                     pe_contiguous=bool(acq[lines[0]:lines[-1] + 1].all()),
                     dc_line=int(np.argmax(m4raw.pe_energy(k))),
                     acquisition=a.get("acquisition", ""), max=a.get("max", np.nan)))
manifest = pd.DataFrame(rows)
manifest.to_csv(paths.meta / "manifest.csv", index=False)

pattern_cols = ["n_slices", "n_coils", "n_fe", "n_pe", "n_pe_acquired", "pe_first",
                "pe_last", "pe_contiguous", "dc_line"]
patterns = manifest.groupby(["contrast"] + pattern_cols).size().rename("files").reset_index()
patterns

**What to look for.** Ideally one row per contrast. For **T2**, `dc_line` must be `n_pe / 2` (128): the masks assume it and notebook 02 checks it. T1 is only ever used as an image, so a T1 centre one line off (127 on the validation subjects) only adds a phase ramp and is harmless.

In [ ]:
ex = {c: manifest[manifest.contrast == c].iloc[0] for c in ("T1", "T2")}
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
for a_, (c, row) in zip(ax, ex.items()):
    path = index[(index.scan_id == row.scan_id) & (index.contrast == c) & (index.rep == row.rep)].path.iloc[0]
    k = m4raw.load_kspace(path, slices=int(row.n_slices) // 2)
    a_.semilogy(m4raw.pe_energy(k) + 1e-12)
    a_.axvline(row.n_pe // 2, color="r", lw=0.8, ls="--", label="n_pe / 2")
    a_.set(title=f"{c}: mean |k| per PE line ({row.n_pe_acquired} acquired)", xlabel="PE line")
    a_.legend()
plt.tight_layout()

## 4. FFT convention check
We recompute the image from k-space with our own `ifft2c` + root-sum-of-squares and compare it with the image stored in the file. A relative error near zero means our FFT convention, axes and orientation match the dataset.

In [ ]:
conv = {}
for c, row in ex.items():
    path = index[(index.scan_id == row.scan_id) & (index.contrast == c) & (index.rep == row.rep)].path.iloc[0]
    ours, stored = kspace_to_rss(m4raw.load_kspace(path)), m4raw.load_rss(path)
    scale = float((ours * stored).sum() / (ours ** 2).sum())    # best-fit global scale
    conv[c] = dict(rel_err=float(np.linalg.norm(ours - stored) / np.linalg.norm(stored)),
                   rel_err_after_scale=float(np.linalg.norm(scale * ours - stored) / np.linalg.norm(stored)),
                   scale=scale)
pd.DataFrame(conv).T

**Reading this.** `rel_err` ≈ 0: identical. `rel_err` large but `rel_err_after_scale` ≈ 0: only a global scale differs, which is harmless. Both large: the geometry differs (shift or flip) and `t1t2/fft.py` must be fixed before continuing.

## 5. Split by subject
We use the official M4Raw folders (train / val / test), which is also the split TGVN released. A subject is usable if it has at least 3 repetitions of both T1 and T2.

In [ ]:
splits = (manifest.pivot_table(index=["split", "scan_id"], columns="contrast", values="rep",
                               aggfunc="count", fill_value=0)
          .rename(columns={"T1": "n_t1", "T2": "n_t2"}).reset_index())
splits["usable"] = (splits.n_t1 >= 3) & (splits.n_t2 >= 3)
splits.to_csv(paths.meta / "splits.csv", index=False)
display(splits.groupby("split").agg(subjects=("scan_id", "size"), usable=("usable", "sum"),
                                    t1_reps=("n_t1", "median"), t2_reps=("n_t2", "median")))

# Cross-check against TGVN's released split files (skipped quietly if offline).
TGVN = "https://raw.githubusercontent.com/sodicksonlab/TGVN/main/data_splits/m4raw/{}.csv"
for split in SPLITS:
    try:
        theirs = set(pd.read_csv(TGVN.format(split)).prefix.str.split("/").str[-1].astype(str))
    except Exception as e:
        print(f"{split}: could not fetch TGVN split ({type(e).__name__})")
        continue
    ours = set(splits[splits.split == split].scan_id)
    if MAX_SUBJECTS:
        print(f"{split}: {len(ours & theirs)}/{len(ours)} of our (subsampled) subjects are in TGVN's {split} list")
    else:
        print(f"{split}: ours {len(ours)}, TGVN {len(theirs)}, only ours {sorted(ours - theirs)}, "
              f"only TGVN {sorted(theirs - ours)}")

## 6. T1–T2 shift per subject
The plan expects about 1.5 px along frequency encoding (B0 drift between the first and last scan). We estimate one sub-pixel translation per subject from edge maps of the repetition-averaged images. `d_fe`, `d_pe` are what must be applied to T1 to line it up with T2.

In [ ]:
def mean_rss(scan_id, contrast, max_reps=3):
    p = index[(index.scan_id == scan_id) & (index.contrast == contrast)].sort_values("rep").path
    return np.mean([m4raw.load_rss(x) for x in p[:max_reps]], axis=0)

rows = []
usable = splits[splits.usable]
for r in tqdm(usable.itertuples(), total=len(usable)):
    t2, t1 = mean_rss(r.scan_id, "T2"), mean_rss(r.scan_id, "T1")
    shift, per_slice = volume_shift(t2, t1)
    rows.append(dict(split=r.split, scan_id=r.scan_id, d_fe=shift[0], d_pe=shift[1],
                     d_fe_slice_std=per_slice[:, 0].std(), d_pe_slice_std=per_slice[:, 1].std()))
alignment = pd.DataFrame(rows)
alignment.to_csv(paths.meta / "alignment.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].hist(alignment.d_fe, bins=30); ax[0].set(title="T1→T2 shift, frequency encoding", xlabel="pixels")
ax[1].hist(alignment.d_pe, bins=30); ax[1].set(title="T1→T2 shift, phase encoding", xlabel="pixels")
plt.tight_layout()
alignment[["d_fe", "d_pe", "d_fe_slice_std", "d_pe_slice_std"]].describe().loc[["mean", "std", "min", "50%", "max"]]

**Reading this.** A large `*_slice_std` for a subject means the slices disagree, so one translation does not describe it (likely real head motion). Those subjects are worth a look in the next cell.

In [ ]:
sid = alignment.iloc[alignment.d_fe.abs().argmax()].scan_id      # the worst-aligned subject
t2, t1 = mean_rss(sid, "T2"), mean_rss(sid, "T1")
s = t2.shape[0] // 2
shift = alignment.set_index("scan_id").loc[sid, ["d_fe", "d_pe"]].to_numpy(float)

def overlay(a, b):   # red = T2 edges, green = T1 edges, yellow = they coincide
    return np.clip(np.dstack([edge_map(a), edge_map(b), np.zeros_like(a)]) * 3, 0, 1)

fig, ax = plt.subplots(1, 4, figsize=(16, 4.3))
ax[0].imshow(t2[s], cmap="gray"); ax[0].set_title("T2 (averaged)")
ax[1].imshow(t1[s], cmap="gray"); ax[1].set_title("T1 (averaged)")
ax[2].imshow(overlay(t2[s], t1[s])); ax[2].set_title("edges before alignment")
ax[3].imshow(overlay(t2[s], apply_shift(t1[s], shift))); ax[3].set_title(f"after shift {shift.round(2)}")
for a_ in ax: a_.axis("off")
plt.suptitle(f"subject {sid}, slice {s}"); plt.tight_layout()

## 7. Are the repetitions phase-stable?
This settles the open "averaging domain" decision. For a few subjects we compare averaging the T2 repetitions as complex images against averaging their magnitudes:

`coherence = |mean of complex repetitions| / mean of |repetitions|`, signal-weighted inside the head.

Close to 1 means the phase is the same in every repetition and complex averaging is safe. We also measure how far T2 itself drifts between its first and last repetition.

In [ ]:
rows = []
for sid in usable.scan_id[:N_PHASE_CHECK]:
    p = index[(index.scan_id == sid) & (index.contrast == "T2")].sort_values("rep").path[:3]
    x = np.stack([ifft2c(m4raw.load_kspace(q)) for q in p])              # [rep, slice, coil, FE, PE]
    mag = np.sqrt((np.abs(x) ** 2).sum(2))                                # RSS per rep
    m = head_mask(mag.mean(0))[:, None]                                  # [slice, 1, FE, PE]
    num = np.abs(x.mean(0)) * m
    den = np.abs(x).mean(0) * m
    drift, _ = volume_shift(mag[0], mag[-1])
    rows.append(dict(scan_id=sid, coherence=float(num.sum() / den.sum()),
                     t2_rep_drift_fe=drift[0], t2_rep_drift_pe=drift[1]))
phase = pd.DataFrame(rows)
phase

In [ ]:
coh = float(phase.coherence.min())
summary = dict(
    smoke=SMOKE, code_commit=commit, n_files=int(len(manifest)),
    subjects={k: int(v) for k, v in splits.groupby("split").usable.sum().items()},
    kspace_patterns=patterns.to_dict("records"),
    fft_convention=conv,
    shift_fe_mean=float(alignment.d_fe.mean()), shift_fe_std=float(alignment.d_fe.std()),
    shift_pe_mean=float(alignment.d_pe.mean()), shift_pe_std=float(alignment.d_pe.std()),
    rep_coherence_min=coh,
    # 0.95 is a rule of thumb, not a published threshold: independent noise alone
    # costs a few percent at single-repetition SNR ~12.
    averaging_domain="complex" if coh > 0.95 else "magnitude",
)
with open(paths.meta / "audit_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)
print(json.dumps({k: v for k, v in summary.items() if k != "kspace_patterns"}, indent=2, default=float))

## Checkpoint
Before moving to `02_preprocess`, confirm:

- [ ] Section 3: one k-space pattern per contrast, and `dc_line` = 128
- [ ] Section 4: `rel_err_after_scale` ≈ 0 for T1 and T2
- [ ] Section 5: subject counts match expectations (128 / 30 / 25) and agree with TGVN
- [ ] Section 6: shift is recorded; the "after" overlay is more yellow than the "before"
- [ ] Section 7: averaging domain noted in the project plan